### BƯỚC 1: Khởi tạo hệ thống
Ô code này sẽ kết nối Qdrant, tải mô hình AI bge-small-en-v1.5 và khởi động lại Spark để làm dữ liệu đối chứng.

In [4]:
import warnings
warnings.filterwarnings('ignore')

from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from pyspark.sql import SparkSession
from datasets import load_dataset
import pandas as pd
import os
import sys

# 1. Khởi tạo Qdrant Client (Kết nối tới database Người 1 đã tạo)
print("🔌 Đang kết nối Qdrant...")
qdrant_client = QdrantClient(url="http://localhost:6333")
collection_name = "Qdrant_Group10"

# 2. Tải mô hình AI (Model này sẽ biến câu search tiếng Anh thành Vector 384 chiều)
print("🤖 Đang tải mô hình AI BAAI/bge-small-en-v1.5...")
model = SentenceTransformer('BAAI/bge-small-en-v1.5')

# 3. Khởi tạo Spark & Tải dữ liệu để LÀM ĐỐI CHỨNG (Giống SQL truyền thống)
print("⚡ Đang khởi tạo Spark để test SQL truyền thống...")
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = SparkSession.builder \
    .appName("HM_Testing_SparkSQL") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("📦 Đang tải dataset vào Spark (Vui lòng đợi vài giây)...")
dataset = load_dataset("Qdrant/hm_ecommerce_products", split="train")
pdf = dataset.to_pandas()[['article_id', 'prod_name', 'detail_desc', 'department_name']]
df_spark = spark.createDataFrame(pdf)

# Đăng ký thành một bảng SQL ảo tên là "hm_products"
df_spark.createOrReplaceTempView("hm_products")

print("✅ HỆ THỐNG ĐÃ SẴN SÀNG CHO PHẦN DEMO & TESTING!")

🔌 Đang kết nối Qdrant...
🤖 Đang tải mô hình AI BAAI/bge-small-en-v1.5...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6628.02it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚡ Đang khởi tạo Spark để test SQL truyền thống...
📦 Đang tải dataset vào Spark (Vui lòng đợi vài giây)...
✅ HỆ THỐNG ĐÃ SẴN SÀNG CHO PHẦN DEMO & TESTING!


### BƯỚC 2: Viết hàm Tìm kiếm Thông minh bằng Qdrant
Hàm này mô phỏng Backend của bạn: Nhận truy vấn người dùng -> Dùng AI biến thành Vector -> Ném vào Qdrant tìm kiếm.

In [7]:
def qdrant_semantic_search(query_text, top_k=3):
    print(f"\n[QDRANT VECTOR SEARCH] Đang tìm kiếm: '{query_text}'")
    print("-" * 80)
    
    # Dùng AI biến câu query thành vector 384 chiều
    query_vector = model.encode(query_text, normalize_embeddings=True).tolist()
    
    search_results = qdrant_client.query_points(
        collection_name=collection_name, # Đảm bảo biến này đã được khai báo ở Bước 1
        query=query_vector,
        limit=top_k
    ).points
    
    for i, hit in enumerate(search_results):
        print(f"Top {i+1} | Độ tương đồng (Score): {hit.score:.4f}")
        print(f"Mã SP (ID): {hit.id}")
        print(f"Tên SP: {hit.payload.get('product_name', 'N/A')}")
        print(f"Phân loại: {hit.payload.get('department_name', 'N/A')}")
        # Lấy 100 ký tự đầu của mô tả để tránh in ra quá dài
        desc = str(hit.payload.get('description', 'N/A'))
        print(f"Mô tả: {desc[:100]}...") 
        print("-" * 80)

### BƯỚC 3: Thực thi Kịch bản Demo (Chạy Test Case theo Yêu cầu)
Chạy 2 Test case được giao trong kịch bản để chứng minh khả năng hiểu Mô tả và Ngữ cảnh/Phong cách.

In [8]:
# Test 1: Tìm kiếm qua công dụng / mô tả
test_1_query = "A warm winter coat for extreme cold weather"
qdrant_semantic_search(test_1_query, top_k=3)

# Test 2: Tìm kiếm theo ngữ cảnh / phong cách
test_2_query = "Comfortable clothes for working out at the gym"
qdrant_semantic_search(test_2_query, top_k=3)


[QDRANT VECTOR SEARCH] Đang tìm kiếm: 'A warm winter coat for extreme cold weather'
--------------------------------------------------------------------------------
Top 1 | Độ tương đồng (Score): 0.7316
Mã SP (ID): 912627002
Tên SP: pro carcoat
Phân loại: Jacket Smart
Mô tả: Car coat in twill with a collar, concealed buttons down the front, diagonal welt front pockets and i...
--------------------------------------------------------------------------------
Top 2 | Độ tương đồng (Score): 0.7315
Mã SP (ID): 765907004
Tên SP: Alfie paletot
Phân loại: Jacket
Mô tả: Single-breasted coat in a wool blend with notch lapels, a chest pocket, flap front pockets and two i...
--------------------------------------------------------------------------------
Top 3 | Độ tương đồng (Score): 0.7315
Mã SP (ID): 765907007
Tên SP: Alfie paletot
Phân loại: Jacket
Mô tả: Single-breasted coat in a wool blend with notch lapels, a chest pocket, flap front pockets and two i...
-----------------------------------

### Khởi động Spark và nạp dữ liệu vào Spark

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from pyspark.sql import SparkSession
from datasets import load_dataset
import pandas as pd
import os
import sys

# 1. Khởi tạo Qdrant Client (Kết nối tới database Người 1 đã tạo)
print("🔌 Đang kết nối Qdrant...")
qdrant_client = QdrantClient(url="http://localhost:6333")
collection_name = "Qdrant_Group10"

# 2. Tải mô hình AI (Model này sẽ biến câu search tiếng Anh thành Vector 384 chiều)
print("🤖 Đang tải mô hình AI BAAI/bge-small-en-v1.5...")
model = SentenceTransformer('BAAI/bge-small-en-v1.5')

# 3. Khởi tạo Spark & Tải dữ liệu để LÀM ĐỐI CHỨNG (Giống SQL truyền thống)
print("⚡ Đang khởi tạo Spark để test SQL truyền thống...")
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = SparkSession.builder \
    .appName("HM_Testing_SparkSQL") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("📦 Đang tải dataset vào Spark (Vui lòng đợi vài giây)...")
dataset = load_dataset("Qdrant/hm_ecommerce_products", split="train")
pdf = dataset.to_pandas()[['article_id', 'prod_name', 'detail_desc', 'department_name']]
df_spark = spark.createDataFrame(pdf)

# Đăng ký thành một bảng SQL ảo tên là "hm_products"
df_spark.createOrReplaceTempView("hm_products")

print("✅ HỆ THỐNG ĐÃ SẴN SÀNG CHO PHẦN DEMO & TESTING!")

🔌 Đang kết nối Qdrant...
🤖 Đang tải mô hình AI BAAI/bge-small-en-v1.5...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6628.02it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚡ Đang khởi tạo Spark để test SQL truyền thống...
📦 Đang tải dataset vào Spark (Vui lòng đợi vài giây)...
✅ HỆ THỐNG ĐÃ SẴN SÀNG CHO PHẦN DEMO & TESTING!


In [9]:
from pyspark.sql.types import StructType, StructField, StringType

print("⏳ Đang ép buộc tắt hệ thống Spark cũ...")
try:
    spark.stop()
except:
    pass

# 1. Khởi tạo lại Spark hoàn toàn mới, cấu hình bộ nhớ thoải mái hơn
print("⚡ Đang khởi động lại Spark (Ultra-Safe Mode)...")
spark = SparkSession.builder \
    .appName("HM_Testing_SparkSQL_Final") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .getOrCreate()

# 2. Xử lý dữ liệu: Ép tất cả về kiểu Chữ (String) và lấp đầy các ô trống
print("⏳ Đang làm sạch dữ liệu...")
columns_to_keep =['article_id', 'prod_name', 'detail_desc', 'department_name']
pdf_clean = pdf[columns_to_keep].fillna("").astype(str)

# 3. BIỆN PHÁP MẠNH: Chuyển Pandas thành List thuần Python để tránh xung đột Worker
print("⏳ Đang đóng gói dữ liệu sang định dạng an toàn...")
data_list = pdf_clean.values.tolist()

# 4. Tự định nghĩa cấu trúc bảng (Schema) cho Spark
schema = StructType([
    StructField("article_id", StringType(), True),
    StructField("prod_name", StringType(), True),
    StructField("detail_desc", StringType(), True),
    StructField("department_name", StringType(), True)
])

# 5. Nạp vào Spark và ép lưu vào RAM (Cache) để truy vấn sau cực nhanh
print("📦 Đang nạp dữ liệu vào Spark...")
df_spark = spark.createDataFrame(data_list, schema=schema)
df_spark.createOrReplaceTempView("hm_products")
df_spark.cache()  # Lưu vào RAM

# Test đếm thử luôn ở đây (để ép Spark chạy thực tế xem còn lỗi không)
total_rows = df_spark.count()
print(f"✅ XUẤT SẮC! Đã nạp thành công {total_rows} dòng dữ liệu vào Spark!")

⏳ Đang ép buộc tắt hệ thống Spark cũ...
⚡ Đang khởi động lại Spark (Ultra-Safe Mode)...
⏳ Đang làm sạch dữ liệu...
⏳ Đang đóng gói dữ liệu sang định dạng an toàn...
📦 Đang nạp dữ liệu vào Spark...
✅ XUẤT SẮC! Đã nạp thành công 105126 dòng dữ liệu vào Spark!


### BƯỚC 4: Thực hiện so sánh (Spark vs Qdrant)
Test Case: Tìm kiếm bằng từ lóng: "kicks for running" (Nghĩa là giày chạy bộ - running shoes)

In [12]:
# Từ lóng "kicks for running" (Nghĩa là giày chạy bộ)
test_3_query = "outfit for sleeping"

print(f"\n{'='*80}")
print(f"PHẦN 1: HỆ THỐNG TRUYỀN THỐNG (SPARK SQL)")
print(f"Đang tìm kiếm: '{test_3_query}'")
print(f"Dùng lệnh SQL: LIKE '%{test_3_query}%' để tìm chuỗi ký tự")
print(f"{'-'*80}")

# Viết truy vấn SQL tìm từ khóa trong Tên sản phẩm hoặc Mô tả
sql_query = f"""
    SELECT prod_name, department_name, detail_desc
    FROM hm_products
    WHERE LOWER(prod_name) LIKE '%{test_3_query}%'
       OR LOWER(detail_desc) LIKE '%{test_3_query}%'
"""

# Thực thi Spark SQL
spark_result = spark.sql(sql_query)

if spark_result.count() == 0:
    print("❌ KẾT QUẢ: 0 SẢN PHẨM TÌM THẤY!")
    print(f"-> Lý do: SQL truyền thống chỉ tìm khớp chuỗi ký tự chính xác. Không có sản phẩm nào miêu tả đúng dòng chữ '{test_3_query}'.")
else:
    spark_result.show(truncate=False)

print(f"\n{'='*80}")
print(f"PHẦN 2: HỆ THỐNG MỚI (QDRANT SEMANTIC SEARCH)")
print(f"{'-'*80}")

# Gọi hàm tìm kiếm bằng vector
qdrant_semantic_search(test_3_query, top_k=3)

# In tổng kết nhận xét
print(f"\n{'*'*80}")
print("🎯 TỔNG KẾT & NHẬN XÉT:")
print("1. Spark SQL thất bại vì tìm kiếm dựa trên so khớp chuỗi (Keyword-based).")
print("2. Qdrant thành công vì tìm kiếm dựa trên khoảng cách Vector trong không gian đa chiều.")
print("   -> Mô hình AI hiểu được ngữ nghĩa 'outfit for sleeping' là đồ mặc khi ngủ, từ đó")
print("      trả về chính xác các trang phục như Pyjamas hay Sleep bag dù văn bản không chứa từ khóa gốc.")
print(f"{'*'*80}")


PHẦN 1: HỆ THỐNG TRUYỀN THỐNG (SPARK SQL)
Đang tìm kiếm: 'outfit for sleeping'
Dùng lệnh SQL: LIKE '%outfit for sleeping%' để tìm chuỗi ký tự
--------------------------------------------------------------------------------
❌ KẾT QUẢ: 0 SẢN PHẨM TÌM THẤY!
-> Lý do: SQL truyền thống chỉ tìm khớp chuỗi ký tự chính xác. Không có sản phẩm nào miêu tả đúng dòng chữ 'outfit for sleeping'.

PHẦN 2: HỆ THỐNG MỚI (QDRANT SEMANTIC SEARCH)
--------------------------------------------------------------------------------

[QDRANT VECTOR SEARCH] Đang tìm kiếm: 'outfit for sleeping'
--------------------------------------------------------------------------------
Top 1 | Độ tương đồng (Score): 0.7503
Mã SP (ID): 557585003
Tên SP: Sleep bag fleece
Phân loại: Baby Nightwear
Mô tả: Sleep bag in patterned fleece with a press-stud on one shoulder and a concealed zip in one side that...
--------------------------------------------------------------------------------
Top 2 | Độ tương đồng (Score): 0.7485
Mã 